# import, paths

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Point

# Set paths
desktop = Path.home() / "Desktop"
output_dir = Path(r"C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step")
output_dir.mkdir(parents=True, exist_ok=True)

print("Libraries imported and paths set")

Libraries imported and paths set


# default_refineries.gpkg

In [4]:
# Prepare default_refineries.gpkg
print("=" * 60)
print("Processing: default_refineries.gpkg")
print("=" * 60)

# Load refineries from desktop
refineries_path = desktop / "refineries_afrec_2020.gpkg"
print(f"\nLoading from: {refineries_path}")

gdf = gpd.read_file(refineries_path)
print(f"Loaded {len(gdf)} refineries")
print(f"Original columns: {gdf.columns.tolist()}")

# Step 1: Rename LPG_OUTPUT_TPY to LPG_prod and Refinery to name
if 'LPG_OUTPUT_TPY' in gdf.columns:
    gdf = gdf.rename(columns={'LPG_OUTPUT_TPY': 'LPG_prod'})
    print("✓ Renamed LPG_OUTPUT_TPY → LPG_prod")
if 'Refinery' in gdf.columns:
    gdf = gdf.rename(columns={'Refinery': 'name'})
    print("✓ Renamed Refinery → name")

# Step 2: Create geometry from lat/lon if they exist and geometry doesn't use them
if 'lat' in gdf.columns and 'lon' in gdf.columns:
    # Create Point geometry from lat/lon
    gdf['geometry'] = gdf.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
    print("✓ Created geometry from lat/lon coordinates")
    
    # Step 3: Remove lat and lon columns
    gdf = gdf.drop(columns=['lat', 'lon'])
    print("✓ Removed lat and lon fields")

# Ensure CRS is set
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
    print("✓ Set CRS to EPSG:4326")

# Save as default_refineries.gpkg
output_path = output_dir / "default_refineries.gpkg"
gdf.to_file(output_path, driver="GPKG")
print(f"\n✓ Saved to: {output_path}")
print(f"Final columns: {gdf.columns.tolist()}")
print(f"Final shape: {len(gdf)} records")

Processing: default_refineries.gpkg

Loading from: C:\Users\Fra\Desktop\refineries_afrec_2020.gpkg
Loaded 22 refineries
Original columns: ['Country', 'Refinery', 'Owner', 'Distillation capacity (Kb/d)', 'LPG_OUTPUT_TPY', 'source', 'lat', 'lon', 'geometry']
✓ Renamed LPG_OUTPUT_TPY → LPG_prod
✓ Renamed Refinery → name
✓ Created geometry from lat/lon coordinates
✓ Removed lat and lon fields

✓ Saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\default_refineries.gpkg
Final columns: ['Country', 'name', 'Owner', 'Distillation capacity (Kb/d)', 'LPG_prod', 'source', 'geometry']
Final shape: 22 records


# default_ports.gpkg

In [5]:
# Prepare default_ports.gpkg
print("\n" + "=" * 60)
print("Processing: default_ports.gpkg")
print("=" * 60)

# Load ports from Tesi_desk/africa_maritime_network
ports_path = desktop / "Tesi_desk" / "africa_maritime_network.gpkg"
print(f"\nLoading from: {ports_path}")

gdf = gpd.read_file(ports_path, layer="nodes_rank")
print(f"Loaded {len(gdf)} ports")
print(f"Original columns: {gdf.columns.tolist()}")

# Step 1: Remove lpg_compliance field if it exists
if 'lpg_compliance' in gdf.columns:
    gdf = gdf.drop(columns=['lpg_compliance'])
    print("✓ Removed lpg_compliance field")

# Step 2: Add VLGC_compliance boolean field based on LPG_max_draft
if 'LPG_max_draft' in gdf.columns:
    gdf['VLGC_compliance'] = pd.to_numeric(gdf['LPG_max_draft'], errors='coerce') > 13
    print("✓ Added VLGC_compliance field (True if LPG_max_draft > 13)")
else:
    gdf['VLGC_compliance'] = False
    print("⚠ LPG_max_draft not found, setting VLGC_compliance to False")

# Step 3: Delete specified fields
fields_to_delete = ['type', 'has_nearby_tank', 'draft_true', 'dwt_l_true', 'dwt_vl_true', 'length_true', 'infra']
existing_fields = [col for col in fields_to_delete if col in gdf.columns]
if existing_fields:
    gdf = gdf.drop(columns=existing_fields)
    print(f"✓ Deleted fields: {', '.join(existing_fields)}")

# Ensure CRS is set
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
    print("✓ Set CRS to EPSG:4326")

# Save as default_ports.gpkg
output_path = output_dir / "default_ports.gpkg"
gdf.to_file(output_path, driver="GPKG")
print(f"\n✓ Saved to: {output_path}")
print(f"Final columns: {gdf.columns.tolist()}")
print(f"Final shape: {len(gdf)} records")
print(f"VLGC_compliance distribution: True={gdf['VLGC_compliance'].sum()}, False={(~gdf['VLGC_compliance']).sum()}")


Processing: default_ports.gpkg

Loading from: C:\Users\Fra\Desktop\Tesi_desk\africa_maritime_network.gpkg
Loaded 118 ports
Original columns: ['id', 'name', 'fullname', 'infra', 'country', 'iso3', 'vessel_count_total', 'vessel_count_container', 'vessel_count_drybulk', 'vessel_count_generalcargo', 'vessel_count_roro', 'vessel_count_tanker', 'industry_top1', 'industry_top2', 'industry_top3', 'share_country_maritime_import', 'share_country_maritime_export', 'call_container', 'call_drybulk', 'call_generalcargo', 'call_roro', 'call_tanker', 'capacity_container_tons', 'capacity_drybulk_tons', 'capacity_generalcargo_tons', 'capacity_roro_tons', 'capacity_tanker_tons', 'time_container_h', 'time_drybulk_h', 'time_generalcargo_h', 'time_roro_h', 'time_tanker_h', 'source', 'type', 'has_nearby_tank', 'LPG_berths', 'LPG_max_draft', 'LPG_max_length', 'LPG_max_DWT', 'draft_true', 'dwt_l_true', 'dwt_vl_true', 'length_true', 'lpg_compliance', 'geometry']
✓ Removed lpg_compliance field
✓ Added VLGC_comp

# default_primary_storage.gpkg

In [6]:
# Prepare default_primary_storage.gpkg
print("\n" + "=" * 60)
print("Processing: default_primary_storage.gpkg")
print("=" * 60)

# Load tank terminals from Tesi_desk
storage_path = desktop / "Tesi_desk" / "tank_terminal_postgmaps_147.gpkg"
print(f"\nLoading from: {storage_path}")

gdf = gpd.read_file(storage_path)
print(f"Loaded {len(gdf)} storage terminals")
print(f"Original columns: {gdf.columns.tolist()}")

# Step 1: Rename tank_capacity to LPG_capacity
if 'tank_capacity' in gdf.columns:
    gdf = gdf.rename(columns={'tank_capacity': 'LPG_capacity'})
    print("✓ Renamed tank_capacity → LPG_capacity")

# Step 2: Remove specified fields
fields_to_delete = ['harbour', 'port_1km', 'port_2km', 'port_3km', 'port_5km', 'port_10km', 'port_20km', 'inland', 'gmaps_opening_hours', 'gmaps_status']
existing_fields = [col for col in fields_to_delete if col in gdf.columns]
if existing_fields:
    gdf = gdf.drop(columns=existing_fields)
    print(f"✓ Deleted fields: {', '.join(existing_fields)}")

# Ensure CRS is set
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
    print("✓ Set CRS to EPSG:4326")

# Save as default_primary_storage.gpkg
output_path = output_dir / "default_primary_storage.gpkg"
gdf.to_file(output_path, driver="GPKG")
print(f"\n✓ Saved to: {output_path}")
print(f"Final columns: {gdf.columns.tolist()}")
print(f"Final shape: {len(gdf)} records")


Processing: default_primary_storage.gpkg

Loading from: C:\Users\Fra\Desktop\Tesi_desk\tank_terminal_postgmaps_147.gpkg
Loaded 147 storage terminals
Original columns: ['name', 'city', 'tanks', 'tank_capacity', 'harbour', 'berths', 'max_draft', 'max_length', 'max_DWT', 'link', 'nearest_port_name', 'nearest_port_distance_m', 'port_1km', 'port_2km', 'port_3km', 'port_5km', 'port_10km', 'port_20km', 'inland', 'error', 'country', 'gmaps_place_id', 'gmaps_name', 'gmaps_keywords', 'gmaps_opening_hours', 'gmaps_status', 'geometry']
✓ Renamed tank_capacity → LPG_capacity
✓ Deleted fields: harbour, port_1km, port_2km, port_3km, port_5km, port_10km, port_20km, inland, gmaps_opening_hours, gmaps_status

✓ Saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\default_primary_storage.gpkg
Final columns: ['name', 'city', 'tanks', 'LPG_capacity', 'berths', 'max_draft', 'max_length', 'max_DWT', 'link', 'nearest_port_name', 'nearest_port_distance_m', 'error', 'country',

# merge default

In [7]:
# Merge all default_*.gpkg files into one multi-layer default.gpkg
print("\n" + "=" * 60)
print("Creating: default.gpkg (4 layers)")
print("=" * 60)

layer_sources = [
    ("refineries", output_dir / "default_refineries.gpkg"),
    ("ports", output_dir / "default_ports.gpkg"),
    ("gas_plants", output_dir / "default_gas_plants.gpkg"),
    ("primary_storage", output_dir / "default_primary_storage.gpkg"),
]

missing = [str(path) for _, path in layer_sources if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing input GPKG files:\n- " + "\n- ".join(missing)
    )

output_path = output_dir / "default.gpkg"
if output_path.exists():
    output_path.unlink()
    print(f"Overwriting existing file: {output_path}")

for i, (layer_name, src_path) in enumerate(layer_sources):
    gdf = gpd.read_file(src_path)
    mode = "w" if i == 0 else "a"
    gdf.to_file(output_path, layer=layer_name, driver="GPKG", mode=mode)
    print(f"✓ Added layer '{layer_name}' from {src_path.name} ({len(gdf)} records)")

print(f"\n✓ Saved merged GPKG: {output_path}")
print("Layers in output:")
for layer_name, _ in layer_sources:
    n = len(gpd.read_file(output_path, layer=layer_name))
    print(f"  - {layer_name}: {n} records")


Creating: default.gpkg (4 layers)


FileNotFoundError: Missing input GPKG files:
- C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\default_gas_plants.gpkg

# perpare lpg_import_cost csv (NOW USELESS...)

In [ ]:
# Prepare lpg_import_cost.csv
print("\n" + "=" * 60)
print("Creating: lpg_import_cost.csv")
print("=" * 60)

import_cost_data = {
    'import_land': [0.019333],
    'import_sea': [0.094167]
}
lpg_import_cost_df = pd.DataFrame(import_cost_data).T

output_path = output_dir / "lpg_import_cost.csv"
lpg_import_cost_df.to_csv(output_path, header=False)
print(f"✓ Saved to: {output_path}")
print(lpg_import_cost_df)

# Prepare national_shares.csv
print("\n" + "=" * 60)
print("Creating: national_shares.csv")
print("=" * 60)

national_shares_data = {
    'ports': [50],
    'refineries': [0.0],
    'gas_plants': [50]
}
national_shares_df = pd.DataFrame(national_shares_data).T

output_path = output_dir / "national_shares.csv"
national_shares_df.to_csv(output_path, header=False)
print(f"✓ Saved to: {output_path}")
print(national_shares_df)


Creating: lpg_import_cost.csv
✓ Saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\lpg_import_cost.csv
                    0
import_land  0.019333
import_sea   0.094167

Creating: national_shares.csv
✓ Saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\national_shares.csv
               0
ports       50.0
refineries   0.0
gas_plants  50.0


# prepare lpg_import_cost.GPKG (by sea) 

In [ ]:
# Create sea transport cost raster from UNCTAD data
print("\n" + "=" * 60)
print("Creating: Transport Cost Raster")
print("=" * 60)

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np
import urllib.request
import zipfile
from pathlib import Path

# Step 1: Load UNCTAD transport cost CSV
transport_cost_path = desktop / "transport_cost_usd-kg_unctad.csv"
print(f"\nLoading CSV from: {transport_cost_path}")

transport_df = pd.read_csv(transport_cost_path, index_col=0)
print(f"Loaded {len(transport_df)} countries")

# Identify the average column
avg_col = [col for col in transport_df.columns if 'average' in col.lower()]
if avg_col:
    avg_col = avg_col[0]
    print(f"Using column: {avg_col}")
else:
    avg_col = transport_df.columns[-1]

# Step 2: Load or download country boundaries
print("\nLoading country boundaries...")
cache_dir = Path.home() / ".geopandas_cache"
cache_dir.mkdir(exist_ok=True)
shp_file = cache_dir / "ne_10m_admin_0_countries.shp"

if not shp_file.exists():
    print("  Downloading Natural Earth country boundaries (10m resolution)...")
    try:
        zip_path = cache_dir / "ne_countries_10m.zip"
        ne_url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
        urllib.request.urlretrieve(ne_url, zip_path)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(cache_dir)
        print("  ✓ Downloaded (10m resolution)")
    except Exception as e:
        print(f"  Error: {e}")
        shp_file = None
else:
    print("  Using cached country boundaries")

# Load world boundaries
if shp_file and shp_file.exists():
    world = gpd.read_file(shp_file)
    world = world.to_crs("EPSG:4326")
    print(f"✓ Loaded {len(world)} countries")
    name_col = 'NAME' if 'NAME' in world.columns else 'name'
else:
    print("ERROR: Could not load country boundaries")
    world = None

# Step 3: Join transport costs to country geometries
if world is not None:
    # Create a mapping dataframe with transport costs
    cost_data = transport_df[[avg_col]].reset_index()
    cost_data.columns = ['country_name', 'transport_cost']
    cost_data['transport_cost'] = pd.to_numeric(cost_data['transport_cost'], errors='coerce')
    
    # Country name mapping for common naming variations (English, Italian, etc.)
    country_name_mapping = {
        # English variations
        'Ivory Coast': 'Côte d\'Ivoire',
        'Cote d\'Ivoire': 'Côte d\'Ivoire',
        'DRC': 'Dem. Rep. Congo',
        'Democratic Republic of Congo': 'Dem. Rep. Congo',
        'Dem. Rep. Congo': 'Dem. Rep. Congo',
        'Congo': 'Congo',
        'Central African Republic': 'Central African Rep.',
        'Central Afr': 'Central African Rep.',
        'Equatorial Guinea': 'Eq. Guin.',
        'Guinea-Bissau': 'Guinea Bissau',
        'Mauritius': 'Mauritius',
        'Madagascar': 'Madagascar',
        'Sao Tome': 'São Tomé and Principe',
        'Sao Tome and Principe': 'São Tomé and Principe',
        
        # Italian names (Sub-Saharan African countries)
        'Angola': 'Angola',
        'Benin': 'Benin',
        'Botswana': 'Botswana',
        'Burkina Faso': 'Burkina Faso',
        'Burundi': 'Burundi',
        'Camerun': 'Cameroon',
        'Cameroon': 'Cameroon',
        'Capo Verde': 'Cape Verde',
        'Cabo Verde': 'Cape Verde',
        'Repubblica Centrafricana': 'Central African Rep.',
        'Ciad': 'Chad',
        'Chad': 'Chad',
        'Comore': 'Comoros',
        'Comoros': 'Comoros',
        'Congo': 'Congo',
        'Repubblica Democratica del Congo': 'Dem. Rep. Congo',
        'Costa d\'Avorio': 'Côte d\'Ivoire',
        'Isola d\'Avorio': 'Côte d\'Ivoire',
        'Gibuti': 'Djibouti',
        'Djibouti': 'Djibouti',
        'Guinea Equatoriale': 'Eq. Guin.',
        'Eritrea': 'Eritrea',
        'Eswatini': 'Eswatini',
        'Swaziland': 'Eswatini',
        'Etiopia': 'Ethiopia',
        'Ethiopia': 'Ethiopia',
        'Gabon': 'Gabon',
        'Gambia': 'Gambia',
        'Ghana': 'Ghana',
        'Guinea': 'Guinea',
        'Guinea-Bissau': 'Guinea Bissau',
        'Guinea Bissau': 'Guinea Bissau',
        'Kenya': 'Kenya',
        'Lesotho': 'Lesotho',
        'Liberia': 'Liberia',
        'Malawi': 'Malawi',
        'Mali': 'Mali',
        'Mauritania': 'Mauritania',
        'Mauritius': 'Mauritius',
        'Madagascar': 'Madagascar',
        'Mozambico': 'Mozambique',
        'Mozambique': 'Mozambique',
        'Namibia': 'Namibia',
        'Niger': 'Niger',
        'Nigeria': 'Nigeria',
        'Ruanda': 'Rwanda',
        'Rwanda': 'Rwanda',
        'São Tomé e Príncipe': 'São Tomé and Principe',
        'Sao Tome e Principe': 'São Tomé and Principe',
        'Senegal': 'Senegal',
        'Seicelle': 'Seychelles',
        'Seychelles': 'Seychelles',
        'Sierra Leone': 'Sierra Leone',
        'Somalia': 'Somalia',
        'Sudafrica': 'South Africa',
        'South Africa': 'South Africa',
        'Sud Sudan': 'S. Sudan',
        'South Sudan': 'S. Sudan',
        'Sudan': 'Sudan',
        'Tanzania': 'Tanzania',
        'Togo': 'Togo',
        'Uganda': 'Uganda',
        'Zambia': 'Zambia',
        'Zimbabwe': 'Zimbabwe',
    }
    
    # Additional mapping for Natural Earth specific names
    ne_name_mapping = {
        'Côte d\'Ivoire': 'Côte d\'Ivoire',
        'Central African Rep.': 'Central African Rep.',
        'Dem. Rep. Congo': 'Dem. Rep. Congo',
        'São Tomé and Principe': 'São Tomé and Principe',
        'Eq. Guin.': 'Eq. Guinea',  # Natural Earth uses "Eq. Guinea"
        'Guinea Bissau': 'Guinea-Bissau',  # Natural Earth uses hyphenated
        'Republic of the Congo': 'Congo',  # Natural Earth uses short form
        'Democratic Republic of Congo': 'Dem. Rep. Congo',  # DRC
    }
    
    # Match country names (case-insensitive with mapping)
    def find_country_in_world(country_name):
        # Step 1: Apply CSV→standard mapping
        standard_name = country_name_mapping.get(country_name, country_name)
        
        # Step 2: Apply standard→Natural Earth mapping
        ne_name = ne_name_mapping.get(standard_name, standard_name)
        
        # Step 3: Try exact match with NE name (prioritize this)
        for idx, row in world.iterrows():
            if str(row[name_col]).lower() == ne_name.lower():
                return idx
        
        # Step 4: Try exact match with standard name
        for idx, row in world.iterrows():
            if str(row[name_col]).lower() == standard_name.lower():
                return idx
        
        # Step 5: Try exact match with original name
        for idx, row in world.iterrows():
            if str(row[name_col]).lower() == country_name.lower():
                return idx
        
        # Step 6: Try whole-word partial matches (avoid substring matches)
        # Don't just check if word is contained - check for word boundaries
        all_names = [country_name, standard_name, ne_name]
        for test_name in all_names:
            test_words = test_name.lower().split()
            for idx, row in world.iterrows():
                world_name = str(row[name_col]).lower()
                # For multi-word country names, try matching multiple words
                if all(word in world_name for word in test_words if len(word) > 3):
                    return idx
        
        return None
    
    # Add transport cost to world dataframe
    world['transport_cost'] = np.nan
    for idx, row in cost_data.iterrows():
        country_idx = find_country_in_world(row['country_name'])
        if country_idx is not None:
            world.at[country_idx, 'transport_cost'] = row['transport_cost']
    
    print(f"✓ Matched countries to transport costs")
    
    # Step 4: Filter to ONLY Sub-Saharan African countries
    print("\nFiltering to Sub-Saharan African countries...")
    
    # Updated list to include all variations and NE names
    sub_saharan_countries = [
        'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 
        'Cape Verde', 'Cabo Verde', 'Central African Rep.', 'Central African Republic',
        'Chad', 'Comoros', 'Congo', 'Dem. Rep. Congo', 'Democratic Republic of the Congo',
        'Cote d\'Ivoire', 'Côte d\'Ivoire', 'Ivory Coast', 'Djibouti', 'Equatorial Guinea', 'Eq. Guin.',
        'Eritrea', 'Eswatini', 'Swaziland', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 
        'Guinea', 'Guinea-Bissau', 'Guinea Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Malawi', 
        'Mali', 'Mauritania', 'Mauritius', 'Madagascar', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 
        'Rwanda', 'Sao Tome and Principe', 'Sao Tome', 'São Tomé and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 
        'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Tanzania', 'Togo', 
        'Uganda', 'Zambia', 'Zimbabwe'
    ]
    
    # Set transport_cost to NaN for non-Sub-Saharan countries
    for idx, row in world.iterrows():
        country_name = str(row[name_col])
        is_sub_saharan = any(
            country_name.lower() == ssa.lower() or 
            ssa.lower() in country_name.lower() 
            for ssa in sub_saharan_countries
        )
        if not is_sub_saharan:
            world.at[idx, 'transport_cost'] = np.nan
    
    print(f"✓ Filtered: keeping only Sub-Saharan African countries")
    print(f"✓ Countries with valid transport costs: {world['transport_cost'].notna().sum()}")
    
    # Step 5: Keep NaN for non-Sub-Saharan countries (no data fill)
    # Only fill NaN for matched Sub-Saharan countries that didn't have a value
    # First, identify which countries are in the Sub-Saharan list
    world['is_sub_saharan'] = world[name_col].apply(
        lambda x: any(
            str(x).lower() == ssa.lower() or 
            ssa.lower() in str(x).lower() 
            for ssa in sub_saharan_countries
        )
    )
    
    # For Sub-Saharan countries without a match, fill with global average
    global_avg = transport_df[avg_col].mean()
    world.loc[world['is_sub_saharan'] & world['transport_cost'].isna(), 'transport_cost'] = global_avg
    
    print(f"✓ Filled Sub-Saharan African countries without data with global average: ${global_avg:.4f}/kg")
    
    # Step 5: Create raster by rasterizing country polygons
    print("\nCreating raster from country boundaries...")
    
    # Define raster bounds and resolution
    bounds = (-20, -35, 55, 37)  # (minx, miny, maxx, maxy)
    resolution = 0.1  # 0.1 degree resolution (~10km)
    width = int((bounds[2] - bounds[0]) / resolution)
    height = int((bounds[3] - bounds[1]) / resolution)
    
    print(f"Raster dimensions: {width} x {height} pixels at {resolution}° resolution")
    
    # Create transform
    transform = from_bounds(*bounds, width, height)
    
    # Prepare geometries with their values for rasterization
    shapes = []
    for idx, row in world.iterrows():
        if pd.notna(row['transport_cost']) and row['geometry'] is not None:
            shapes.append((row['geometry'], float(row['transport_cost'])))
    
    print(f"Rasterizing {len(shapes)} country geometries...")
    
    # Rasterize: burn country boundaries with their transport cost values
    raster = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype=np.float32
    )
    
    # Save raster
    output_path = output_dir / "transport_cost_sea_raster.tif"
    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=raster.dtype,
        crs='EPSG:4326',
        transform=transform
    ) as dst:
        dst.write(raster, 1)
    
    print(f"\n✓ Raster saved to: {output_path}")
    print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")
    valid_data = raster[~np.isnan(raster)]
    if len(valid_data) > 0:
        print(f"  Transport cost range: ${valid_data.min():.4f} - ${valid_data.max():.4f}/kg")



Creating: Transport Cost Raster

Loading CSV from: C:\Users\Fra\Desktop\transport_cost_usd-kg_unctad.csv
Loaded 38 countries
Using column: average

Loading country boundaries...
  Using cached country boundaries
✓ Loaded 258 countries
✓ Matched countries to transport costs

Filtering to Sub-Saharan African countries...
✓ Filtered: keeping only Sub-Saharan African countries
✓ Countries with valid transport costs: 38
✓ Filled Sub-Saharan African countries without data with global average: $0.1479/kg

Creating raster from country boundaries...
Raster dimensions: 750 x 720 pixels at 0.1° resolution
Rasterizing 51 country geometries...

✓ Raster saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\transport_cost_sea_raster.tif
  File size: 2.1 MB
  Transport cost range: $0.0403 - $1.2535/kg


# prepare lpg_import_cost.GPKG (by road)

In [ ]:
# Create road transport cost raster from UNCTAD data
print("\n" + "=" * 60)
print("Creating: Road Transport Cost Raster")
print("=" * 60)

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np
import urllib.request
import zipfile
from pathlib import Path

# Step 1: Load UNCTAD transport cost CSV
transport_cost_path = desktop / "transport_cost_road_unctad.csv"
print(f"\nLoading CSV from: {transport_cost_path}")

transport_df = pd.read_csv(transport_cost_path, index_col=0)
print(f"Loaded {len(transport_df)} countries")

# Identify the average column
avg_col = [col for col in transport_df.columns if 'average' in col.lower()]
if avg_col:
    avg_col = avg_col[0]
    print(f"Using column: {avg_col}")
else:
    avg_col = transport_df.columns[-1]

# Step 2: Load or download country boundaries
print("\nLoading country boundaries...")
cache_dir = Path.home() / ".geopandas_cache"
cache_dir.mkdir(exist_ok=True)
shp_file = cache_dir / "ne_10m_admin_0_countries.shp"

if not shp_file.exists():
    print("  Downloading Natural Earth country boundaries (10m resolution)...")
    try:
        zip_path = cache_dir / "ne_countries_10m.zip"
        ne_url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
        urllib.request.urlretrieve(ne_url, zip_path)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(cache_dir)
        print("  ✓ Downloaded (10m resolution)")
    except Exception as e:
        print(f"  Error: {e}")
        shp_file = None
else:
    print("  Using cached country boundaries")

# Load world boundaries
if shp_file and shp_file.exists():
    world = gpd.read_file(shp_file)
    world = world.to_crs("EPSG:4326")
    print(f"✓ Loaded {len(world)} countries")
    name_col = 'NAME' if 'NAME' in world.columns else 'name'
else:
    print("ERROR: Could not load country boundaries")
    world = None

# Step 3: Join transport costs to country geometries
if world is not None:
    # Create a mapping dataframe with transport costs
    cost_data = transport_df[[avg_col]].reset_index()
    cost_data.columns = ['country_name', 'transport_cost']
    cost_data['transport_cost'] = pd.to_numeric(cost_data['transport_cost'], errors='coerce')
    
    # Country name mapping for common naming variations (English, Italian, etc.)
    country_name_mapping = {
        # English variations
        'Ivory Coast': 'Côte d\'Ivoire',
        'Cote d\'Ivoire': 'Côte d\'Ivoire',
        'DRC': 'Dem. Rep. Congo',
        'Democratic Republic Of Congo': 'Dem. Rep. Congo',
        'Dem. Rep. Congo': 'Dem. Rep. Congo',
        'Congo': 'Congo',
        'Central African Republic': 'Central African Rep.',
        'Central Afr': 'Central African Rep.',
        'Equatorial Guinea': 'Eq. Guin.',
        'Guinea-Bissau': 'Guinea Bissau',
        'Mauritius': 'Mauritius',
        'Madagascar': 'Madagascar',
        'Sao Tome': 'São Tomé and Principe',
        'Sao Tome and Principe': 'São Tomé and Principe',
        
        # Italian names (Sub-Saharan African countries)
        'Angola': 'Angola',
        'Benin': 'Benin',
        'Botswana': 'Botswana',
        'Burkina Faso': 'Burkina Faso',
        'Burundi': 'Burundi',
        'Camerun': 'Cameroon',
        'Cameroon': 'Cameroon',
        'Capo Verde': 'Cape Verde',
        'Cabo Verde': 'Cape Verde',
        'Repubblica Centrafricana': 'Central African Rep.',
        'Ciad': 'Chad',
        'Chad': 'Chad',
        'Comore': 'Comoros',
        'Comoros': 'Comoros',
        'Congo': 'Congo',
        'Repubblica Democratica del Congo': 'Dem. Rep. Congo',
        'Costa d\'Avorio': 'Côte d\'Ivoire',
        'Isola d\'Avorio': 'Côte d\'Ivoire',
        'Gibuti': 'Djibouti',
        'Djibouti': 'Djibouti',
        'Guinea Equatoriale': 'Eq. Guin.',
        'Eritrea': 'Eritrea',
        'Eswatini': 'Eswatini',
        'Swaziland': 'Eswatini',
        'Etiopia': 'Ethiopia',
        'Ethiopia': 'Ethiopia',
        'Gabon': 'Gabon',
        'Gambia': 'Gambia',
        'Ghana': 'Ghana',
        'Guinea': 'Guinea',
        'Guinea-Bissau': 'Guinea Bissau',
        'Guinea Bissau': 'Guinea Bissau',
        'Kenya': 'Kenya',
        'Lesotho': 'Lesotho',
        'Liberia': 'Liberia',
        'Malawi': 'Malawi',
        'Mali': 'Mali',
        'Mauritania': 'Mauritania',
        'Mauritius': 'Mauritius',
        'Madagascar': 'Madagascar',
        'Mozambico': 'Mozambique',
        'Mozambique': 'Mozambique',
        'Namibia': 'Namibia',
        'Niger': 'Niger',
        'Nigeria': 'Nigeria',
        'Ruanda': 'Rwanda',
        'Rwanda': 'Rwanda',
        'São Tomé e Príncipe': 'São Tomé and Principe',
        'Sao Tome e Principe': 'São Tomé and Principe',
        'Senegal': 'Senegal',
        'Seicelle': 'Seychelles',
        'Seychelles': 'Seychelles',
        'Sierra Leone': 'Sierra Leone',
        'Somalia': 'Somalia',
        'Sudafrica': 'South Africa',
        'South Africa': 'South Africa',
        'Sud Sudan': 'S. Sudan',
        'South Sudan': 'S. Sudan',
        'Sudan': 'Sudan',
        'Tanzania': 'Tanzania',
        'Togo': 'Togo',
        'Uganda': 'Uganda',
        'Zambia': 'Zambia',
        'Zimbabwe': 'Zimbabwe',
    }
    
    # Additional mapping for Natural Earth specific names
    ne_name_mapping = {
        'Côte d\'Ivoire': 'Côte d\'Ivoire',
        'Central African Rep.': 'Central African Rep.',
        'Dem. Rep. Congo': 'Dem. Rep. Congo',
        'São Tomé and Principe': 'São Tomé and Principe',
        'Eq. Guin.': 'Eq. Guinea',  # Natural Earth uses "Eq. Guinea"
        'Guinea Bissau': 'Guinea-Bissau',  # Natural Earth uses hyphenated
        'Republic of the Congo': 'Congo',  # Natural Earth uses short form
        'Democratic Republic of Congo': 'Dem. Rep. Congo',  # DRC
    }
    
    # Match country names (case-insensitive with mapping)
    def find_country_in_world(country_name):
        # Step 1: Apply CSV→standard mapping
        standard_name = country_name_mapping.get(country_name, country_name)
        
        # Step 2: Apply standard→Natural Earth mapping
        ne_name = ne_name_mapping.get(standard_name, standard_name)
        
        # Step 3: Try exact match with NE name (prioritize this)
        for idx, row in world.iterrows():
            if str(row[name_col]).lower() == ne_name.lower():
                return idx
        
        # Step 4: Try exact match with standard name
        for idx, row in world.iterrows():
            if str(row[name_col]).lower() == standard_name.lower():
                return idx
        
        # Step 5: Try exact match with original name
        for idx, row in world.iterrows():
            if str(row[name_col]).lower() == country_name.lower():
                return idx
        
        # Step 6: Try whole-word partial matches (avoid substring matches)
        # Don't just check if word is contained - check for word boundaries
        all_names = [country_name, standard_name, ne_name]
        for test_name in all_names:
            test_words = test_name.lower().split()
            for idx, row in world.iterrows():
                world_name = str(row[name_col]).lower()
                # For multi-word country names, try matching multiple words
                if all(word in world_name for word in test_words if len(word) > 3):
                    return idx
        
        return None
    
    # Add transport cost to world dataframe
    world['transport_cost'] = np.nan
    for idx, row in cost_data.iterrows():
        country_idx = find_country_in_world(row['country_name'])
        if country_idx is not None:
            world.at[country_idx, 'transport_cost'] = row['transport_cost']
    
    print(f"✓ Matched countries to transport costs")
    
    # Step 4: Filter to ONLY Sub-Saharan African countries
    print("\nFiltering to Sub-Saharan African countries...")
    
    # Updated list to include all variations and NE names
    sub_saharan_countries = [
        'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 
        'Cape Verde', 'Cabo Verde', 'Central African Rep.', 'Central African Republic',
        'Chad', 'Comoros', 'Congo', 'Dem. Rep. Congo', 'Democratic Republic of the Congo',
        'Cote d\'Ivoire', 'Côte d\'Ivoire', 'Ivory Coast', 'Djibouti', 'Equatorial Guinea', 'Eq. Guin.',
        'Eritrea', 'Eswatini', 'Swaziland', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 
        'Guinea', 'Guinea-Bissau', 'Guinea Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Malawi', 
        'Mali', 'Mauritania', 'Mauritius', 'Madagascar', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 
        'Rwanda', 'Sao Tome and Principe', 'Sao Tome', 'São Tomé and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 
        'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Tanzania', 'Togo', 
        'Uganda', 'Zambia', 'Zimbabwe'
    ]
    
    # Set transport_cost to NaN for non-Sub-Saharan countries
    for idx, row in world.iterrows():
        country_name = str(row[name_col])
        is_sub_saharan = any(
            country_name.lower() == ssa.lower() or 
            ssa.lower() in country_name.lower() 
            for ssa in sub_saharan_countries
        )
        if not is_sub_saharan:
            world.at[idx, 'transport_cost'] = np.nan
    
    print(f"✓ Filtered: keeping only Sub-Saharan African countries")
    print(f"✓ Countries with valid transport costs: {world['transport_cost'].notna().sum()}")
    
    # Step 5: Keep NaN for non-Sub-Saharan countries (no data fill)
    # Only fill NaN for matched Sub-Saharan countries that didn't have a value
    # First, identify which countries are in the Sub-Saharan list
    world['is_sub_saharan'] = world[name_col].apply(
        lambda x: any(
            str(x).lower() == ssa.lower() or 
            ssa.lower() in str(x).lower() 
            for ssa in sub_saharan_countries
        )
    )
    
    # For Sub-Saharan countries without a match, fill with global average
    global_avg = transport_df[avg_col].mean()
    world.loc[world['is_sub_saharan'] & world['transport_cost'].isna(), 'transport_cost'] = global_avg
    
    print(f"✓ Filled Sub-Saharan African countries without data with global average: ${global_avg:.4f}/kg")
    
    # Step 5: Create raster by rasterizing country polygons
    print("\nCreating raster from country boundaries...")
    
    # Define raster bounds and resolution
    bounds = (-20, -35, 55, 37)  # (minx, miny, maxx, maxy)
    resolution = 0.1  # 0.1 degree resolution (~10km)
    width = int((bounds[2] - bounds[0]) / resolution)
    height = int((bounds[3] - bounds[1]) / resolution)
    
    print(f"Raster dimensions: {width} x {height} pixels at {resolution}° resolution")
    
    # Create transform
    transform = from_bounds(*bounds, width, height)
    
    # Prepare geometries with their values for rasterization
    shapes = []
    for idx, row in world.iterrows():
        if pd.notna(row['transport_cost']) and row['geometry'] is not None:
            shapes.append((row['geometry'], float(row['transport_cost'])))
    
    print(f"Rasterizing {len(shapes)} country geometries...")
    
    # Rasterize: burn country boundaries with their transport cost values
    raster = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype=np.float32
    )
    
    # Save raster
    output_path = output_dir / "transport_cost_road_raster.tif"
    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=raster.dtype,
        crs='EPSG:4326',
        transform=transform
    ) as dst:
        dst.write(raster, 1)
    
    print(f"\n✓ Raster saved to: {output_path}")
    print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")
    valid_data = raster[~np.isnan(raster)]
    if len(valid_data) > 0:
        print(f"  Transport cost range: ${valid_data.min():.4f} - ${valid_data.max():.4f}/kg")


Creating: Road Transport Cost Raster

Loading CSV from: C:\Users\Fra\Desktop\transport_cost_road_unctad.csv
Loaded 35 countries
Using column: average

Loading country boundaries...
  Using cached country boundaries
✓ Loaded 258 countries
✓ Matched countries to transport costs

Filtering to Sub-Saharan African countries...
✓ Filtered: keeping only Sub-Saharan African countries
✓ Countries with valid transport costs: 35
✓ Filled Sub-Saharan African countries without data with global average: $0.2746/kg

Creating raster from country boundaries...
Raster dimensions: 750 x 720 pixels at 0.1° resolution
Rasterizing 51 country geometries...

✓ Raster saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\transport_cost_road_raster.tif
  File size: 2.1 MB
  Transport cost range: $0.0145 - $2.5750/kg


# join the import cost gpkg in a multi layer gpkg

In [ ]:
# Join sea and road transport cost rasters into one multi-band raster
print("\n" + "=" * 60)
print("Creating: transport_cost_raster (multi-band)")
print("=" * 60)

import rasterio

# Input rasters (fixed names, no fallback)
sea_path = output_dir / "transport_cost_sea_raster.tif"
road_path = output_dir / "transport_cost_road_raster.tif"

if not sea_path.exists():
    raise FileNotFoundError(f"Missing required input raster: {sea_path}")
if not road_path.exists():
    raise FileNotFoundError(f"Missing required input raster: {road_path}")

print(f"Sea raster:  {sea_path.name}")
print(f"Road raster: {road_path.name}")

# Output path
output_path = output_dir / "transport_cost_raster.tif"

# Read input rasters and validate compatibility
with rasterio.open(sea_path) as sea_src, rasterio.open(road_path) as road_src:
    if sea_src.width != road_src.width or sea_src.height != road_src.height:
        raise ValueError("Input rasters have different dimensions")
    if sea_src.transform != road_src.transform:
        raise ValueError("Input rasters have different geotransforms")
    if sea_src.crs != road_src.crs:
        raise ValueError("Input rasters have different CRS")

    sea_data = sea_src.read(1)
    road_data = road_src.read(1)

    # Write 2-band output raster
    profile = sea_src.profile.copy()
    profile.update(count=2)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(sea_data, 1)
        dst.write(road_data, 2)
        dst.update_tags(1, DESCRIPTION="sea")
        dst.update_tags(2, DESCRIPTION="road")

print(f"\n✓ Multi-band raster saved to: {output_path}")
with rasterio.open(output_path) as src:
    print(f"  Bands: {src.count}")
    print(f"  Size: {src.width} x {src.height}")
    print(f"  CRS: {src.crs}")


Creating: transport_cost_raster (multi-band)
Sea raster:  transport_cost_sea_raster.tif
Road raster: transport_cost_road_raster.tif

✓ Multi-band raster saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\transport_cost_raster.tif
  Bands: 2
  Size: 750 x 720
  CRS: EPSG:4326


# creare multi band supply share raster 

In [ ]:
# Create supply share raster (multi-band: ports, refineries, gas_plants)
print("\n" + "=" * 60)
print("Creating: Supply Share Multi-Band Raster")
print("=" * 60)

# Step 1: Load supply share data by country
supply_path = desktop / "LPG_supply_shares.csv"
print(f"\nLoading supply shares from: {supply_path}")

try:
    # Try multiple encodings to handle Italian characters and special characters
    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-15']
    supply_df = None
    
    for encoding in encodings:
        try:
            supply_df = pd.read_csv(supply_path, index_col=0, encoding=encoding)  # Country as index
            print(f"✓ CSV loaded with encoding: {encoding}")
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    if supply_df is None:
        raise ValueError("Could not read CSV with any supported encoding")
    
    print(f"Loaded {len(supply_df)} countries")
    print(f"Columns: {supply_df.columns.tolist()}")
    
    # Ensure we have the three required columns
    required_cols = ['ports', 'refineries', 'gas_plants']
    if all(col in supply_df.columns for col in required_cols):
        print("✓ All required columns found")
    else:
        print(f"⚠ Missing columns. Available: {supply_df.columns.tolist()}")
    
    # Step 2: Initialize 3 bands in world dataframe
    world['supply_ports'] = np.nan
    world['supply_refineries'] = np.nan
    world['supply_gas_plants'] = np.nan
    
    # Step 3: Match supply data to countries
    matched_supply = 0
    for idx, row in supply_df.iterrows():
        country_name = idx
        country_idx = find_country_in_world(country_name)
        if country_idx is not None:
            world.at[country_idx, 'supply_ports'] = float(row.get('ports', 0))
            world.at[country_idx, 'supply_refineries'] = float(row.get('refineries', 0))
            world.at[country_idx, 'supply_gas_plants'] = float(row.get('gas_plants', 0))
            matched_supply += 1
    
    print(f"✓ Matched {matched_supply}/{len(supply_df)} countries")
    
    # Step 4: Fill NaN values for Sub-Saharan countries with 0 (no supply)
    # For countries that don't have supply data explicitly
    world.loc[world['is_sub_saharan'] & world['supply_ports'].isna(), 'supply_ports'] = 0
    world.loc[world['is_sub_saharan'] & world['supply_refineries'].isna(), 'supply_refineries'] = 0
    world.loc[world['is_sub_saharan'] & world['supply_gas_plants'].isna(), 'supply_gas_plants'] = 0
    
    # Step 4b: For areas with 0, 0, 0 (no supply data), assign default: 100% from ports
    zero_supply = (world['supply_ports'] == 0) & (world['supply_refineries'] == 0) & (world['supply_gas_plants'] == 0)
    world.loc[zero_supply, 'supply_ports'] = 100
    
    print(f"✓ Filled missing values with 0")
    print(f"✓ Assigned default supply (100% ports) for {zero_supply.sum()} areas with no supply data")
    
    # Step 5: Create 3-band raster
    print("\nCreating multi-band raster...")
    print(f"Raster dimensions: {width} x {height} pixels")
    
    # Initialize 3 empty arrays for each band
    raster_ports = rasterize(
        [(row['geometry'], float(row['supply_ports'])) for idx, row in world.iterrows() 
         if pd.notna(row['supply_ports']) and row['geometry'] is not None],
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype=np.float32
    )
    
    raster_refineries = rasterize(
        [(row['geometry'], float(row['supply_refineries'])) for idx, row in world.iterrows() 
         if pd.notna(row['supply_refineries']) and row['geometry'] is not None],
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype=np.float32
    )
    
    raster_gas_plants = rasterize(
        [(row['geometry'], float(row['supply_gas_plants'])) for idx, row in world.iterrows() 
         if pd.notna(row['supply_gas_plants']) and row['geometry'] is not None],
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype=np.float32
    )
    
    print("✓ Rasterized all 3 bands")
    
    # Step 6: Save as multi-band GeoTIFF
    output_path = output_dir / "supply_share_multiband.tif"
    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=3,  # 3 bands
        dtype=raster_ports.dtype,
        crs='EPSG:4326',
        transform=transform
    ) as dst:
        dst.write(raster_ports, 1)  # Band 1: ports
        dst.write(raster_refineries, 2)  # Band 2: refineries
        dst.write(raster_gas_plants, 3)  # Band 3: gas_plants
        
        # Add band descriptions using actual column names from CSV
        band_names = supply_df.columns.tolist()  # ['ports', 'refineries', 'gas_plants']
        for band_idx, col_name in enumerate(band_names, 1):
            dst.update_tags(band_idx, long_name=f'{col_name} (%)')
    
    print(f"\n✓ Multi-band raster saved to: {output_path}")
    print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")
    print(f"  Bands: 3 (ports, refineries, gas_plants)")
    print(f"  Format: GeoTIFF with EPSG:4326 CRS")
    
    # Verify bands
    with rasterio.open(output_path) as src:
        print(f"\n  Verification:")
        print(f"    Width: {src.width}, Height: {src.height}")
        print(f"    Number of bands: {src.count}")
        for i in range(1, src.count + 1):
            band_data = src.read(i)
            valid = band_data[~np.isnan(band_data)]
            if len(valid) > 0:
                print(f"    Band {i}: min={valid.min():.1f}%, max={valid.max():.1f}%")

except FileNotFoundError:
    print(f"⚠ File not found: {supply_path}")
    print("Please ensure the supply shares CSV file is on the Desktop")
except Exception as e:
    print(f"⚠ Error: {e}")


Creating: Supply Share Multi-Band Raster

Loading supply shares from: C:\Users\Fra\Desktop\LPG_supply_shares.csv
✓ CSV loaded with encoding: latin-1
Loaded 49 countries
Columns: ['ports', 'refineries', 'gas_plants']
✓ All required columns found
✓ Matched 48/49 countries
✓ Filled missing values with 0
✓ Assigned default supply (100% ports) for 3 areas with no supply data

Creating multi-band raster...
Raster dimensions: 750 x 720 pixels
✓ Rasterized all 3 bands

✓ Multi-band raster saved to: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\supply_share_multiband.tif
  File size: 6.2 MB
  Bands: 3 (ports, refineries, gas_plants)
  Format: GeoTIFF with EPSG:4326 CRS

  Verification:
    Width: 750, Height: 720
    Number of bands: 3
    Band 1: min=0.0%, max=100.0%
    Band 2: min=0.0%, max=100.0%
    Band 3: min=0.0%, max=100.0%


# raster names adjusting

In [ ]:
# Rename bands in the supply share raster to match CSV column names
print("\n" + "=" * 60)
print("Renaming Raster Bands")
print("=" * 60)

import rasterio
from rasterio.io import MemoryFile
import numpy as np

supply_raster_path = output_dir / "supply_share_multiband.tif"
col_names = ['ports', 'refineries', 'gas_plants']

print(f"\nUpdating band names for: {supply_raster_path}")
print(f"Band names: {col_names}")

# Read the current raster and update band descriptions
with rasterio.open(supply_raster_path, 'r+') as src:
    for band_idx, col_name in enumerate(col_names, 1):
        src.update_tags(band_idx, DESCRIPTION=col_name)
        print(f"  Band {band_idx}: '{col_name}'")

print("✓ Band names updated successfully")


Renaming Raster Bands

Updating band names for: C:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\dataset_first_step\supply_share_multiband.tif
Band names: ['ports', 'refineries', 'gas_plants']
  Band 1: 'ports'
  Band 2: 'refineries'
  Band 3: 'gas_plants'
✓ Band names updated successfully
